# Naive Bayes Frequency & Probability Tables

This notebook loads the categorized data to generate the Naive Bayes frequency and probability tables, exports them as a beautifully formatted CSV, and computes class priors.

In [9]:
import pandas as pd
import numpy as np

# Load the categorized training data
X_train_cat = pd.read_csv('../x_train_categorized.csv')
y_train = pd.read_csv('../y_train.csv')

FEATURES = ['hr_mean', 'hr_sdnn_5', 'hr_slope_3', 'rr_mean', 'rr_slope_3', 'minutes_since_start']
TARGET = 'sleep_stage'
STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 5: 'REM'}
CLASS_ORDER = ['Wake', 'N1', 'N2', 'N3', 'REM']

## 1. Table Generation Function

Builds a cross-tabulation of feature categories vs. sleep stages, formatted as counts and string fractions.

In [10]:
def build_naive_bayes_table(feature_col, x_data, y_data):
    """Build a frequency + probability table for one feature."""
    combined = pd.DataFrame({
        'feature': x_data[feature_col],
        'class': y_data.map(STAGE_NAMES)
    })

    # Frequency crosstab
    freq = pd.crosstab(combined['feature'], combined['class'])
    freq = freq.reindex(columns=CLASS_ORDER)

    # Add totals row
    freq.loc['Total'] = freq.sum()

    # Create strings for the probability columns (fraction format)
    prob_cols = []
    class_totals = freq.loc['Total']
    
    for c in CLASS_ORDER:
        prob_series = freq[c].apply(lambda x: f"{int(x)}/{int(class_totals[c])}")
        prob_cols.append(prob_series)
        
    prob_df = pd.concat(prob_cols, axis=1)
    prob_df.columns = [f'P({c})' for c in CLASS_ORDER]
    
    # Combine into one display table
    result = pd.concat([freq, prob_df], axis=1)
    
    # Clear probability cells for the 'Total' row
    for c in CLASS_ORDER:
        result.loc['Total', f'P({c})'] = ""
        
    return result

## 2. Generate and Export CSV

For each feature, we generate the table and write it sequentially into a single beautifully formatted CSV file.

In [11]:
output_csv = 'naive_bayes_tables.csv'

# Clear the file and add a title
with open(output_csv, 'w') as f:
    f.write("Naive Bayes Frequency & Probability Tables\n\n")

for feature in FEATURES:
    print(f"\n{'=' * 90}")
    print(f"  {feature}")
    print(f"{'=' * 90}")
    
    table = build_naive_bayes_table(feature, X_train_cat, y_train['sleep_stage'])
    print(table.to_string())
    
    # Write feature header to CSV
    with open(output_csv, 'a') as f:
        f.write(f"{feature.capitalize()}\n")
        
    # Append the dataframe
    table.to_csv(output_csv, mode='a')
    
    # Add spacing between tables
    with open(output_csv, 'a') as f:
        f.write("\n\n")
        
print(f"\nAll feature tables written to {output_csv}")


  hr_mean
               Wake     N1      N2     N3    REM       P(Wake)        P(N1)         P(N2)        P(N3)       P(REM)
feature                                                                                                            
Bradycardia   19369   6807   56992  13444  16603  19369/123531   6807/18738  56992/159946  13444/44150  16603/53635
Normal        81983  11456   99475  29825  35951  81983/123531  11456/18738  99475/159946  29825/44150  35951/53635
Tachycardia   22179    475    3479    881   1081  22179/123531    475/18738   3479/159946    881/44150   1081/53635
Total        123531  18738  159946  44150  53635                                                                   

  hr_sdnn_5
            Wake     N1      N2     N3    REM       P(Wake)       P(N1)         P(N2)        P(N3)       P(REM)
feature                                                                                                        
High       55202   4311   16251   2499  10454  55202/123

## 3. Class Prior Probabilities

The prior probability of each sleep stage in the training data, appended to the CSV.

In [12]:
print("Class Prior Probabilities P(class) -- Training Data")
print("=" * 50)

total_samples = len(y_train)
priors_counts = y_train['sleep_stage'].value_counts().sort_index()
priors_counts.index = priors_counts.index.map(STAGE_NAMES)

priors_df = pd.DataFrame({
    'Count': priors_counts.values,
    'P(class)': [f"{count}/{total_samples}" for count in priors_counts]
}, index=priors_counts.index)

print(priors_df.to_string())

# Append Priors to CSV
with open(output_csv, 'a') as f:
    f.write("Class Priors\n")
priors_df.to_csv(output_csv, mode='a')

print(f"\nClass Priors appended to {output_csv}")

Class Prior Probabilities P(class) -- Training Data
              Count       P(class)
sleep_stage                       
Wake         123531  123531/400000
N1            18738   18738/400000
N2           159946  159946/400000
N3            44150   44150/400000
REM           53635   53635/400000

Class Priors appended to naive_bayes_tables.csv


## 4. Verification

Confirm that conditional probabilities sum to 1.0 for each class.

In [13]:
print(f'\nP(category|class) sum-to-1 check:')
all_pass = True
for feature in FEATURES:
    table = build_naive_bayes_table(feature, X_train_cat, y_train['sleep_stage'])
    
    # We need to evaluate the fractions to check the sum
    prob_cols = [c for c in table.columns if c.startswith('P(')]
    prob_sums = []
    for col in prob_cols:
        col_sum = sum(eval(frac) for frac in table.loc[table.index != 'Total', col])
        prob_sums.append(col_sum)
        
    ok = all(abs(s - 1.0) < 0.001 for s in prob_sums)
    if not ok:
        all_pass = False
    status = 'PASS' if ok else 'FAIL'
    print(f'  [{status}] {feature}')
print(f'\nAll probability columns sum to 1.0: {"YES" if all_pass else "NO"}')


P(category|class) sum-to-1 check:
  [PASS] hr_mean
  [PASS] hr_sdnn_5
  [PASS] hr_slope_3
  [PASS] rr_mean
  [PASS] rr_slope_3
  [PASS] minutes_since_start

All probability columns sum to 1.0: YES
